# 01 Data Inventory

Structured inventory of candidate FRED and Census sources for the data-readiness phase.

In [1]:
from pathlib import Path
import json, os, requests
import pandas as pd
import numpy as np
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'; RAW_DIR = DATA_DIR / 'raw'; INTERIM_DIR = DATA_DIR / 'interim'; PROCESSED_DIR = DATA_DIR / 'processed'; REPORTS_DIR = ROOT / 'reports'
for p in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, REPORTS_DIR]: p.mkdir(parents=True, exist_ok=True)
def load_local_env():
    env_path = ROOT / '.env'
    if env_path.exists():
        for line in env_path.read_text().splitlines():
            if line.strip() and not line.strip().startswith('#') and '=' in line:
                k, v = line.split('=', 1); os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
try:
    from dotenv import load_dotenv; load_dotenv(ROOT / '.env')
except Exception:
    load_local_env()
def get_api_key(name, required=False):
    value = os.environ.get(name)
    if value: return value
    secrets_path = ROOT / '.secrets' / 'api_keys.json'
    if secrets_path.exists():
        try:
            value = json.loads(secrets_path.read_text()).get(name)
            if value:
                os.environ.setdefault(name, value); return value
        except json.JSONDecodeError: pass
    if required: raise RuntimeError(f'Missing {name}. Set it as an environment variable or in local .env.')
    return None
def write_csv(df, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True); df.to_csv(path, index=False); print(f'wrote {path.relative_to(ROOT)} ({len(df):,} rows)')
FRED_API_KEY = get_api_key('FRED_API_KEY')
CENSUS_API_KEY = get_api_key('CENSUS_API_KEY')

In [2]:
FRED_BASE = 'https://api.stlouisfed.org/fred'
fred_candidates = [
 {'id':'MSPUS','desc':'Median Sales Price of Houses Sold for the United States','role':'Long-run housing sticker-price anchor','priority':'required'},
 {'id':'CPIAUCSL','desc':'Consumer Price Index for All Urban Consumers: All Items','role':'Inflation adjustment baseline','priority':'required'},
 {'id':'MEHOINUSA672N','desc':'Real Median Household Income in the United States','role':'Income / ability-to-pay context','priority':'required'},
 {'id':'MORTGAGE30US','desc':'30-Year Fixed Rate Mortgage Average in the United States','role':'Financing context','priority':'required'},
 {'id':'CSUSHPINSA','desc':'S&P CoreLogic Case-Shiller U.S. National Home Price Index','role':'Repeat-sales home price index context','priority':'required'},
 {'id':'CUSR0000SAH1','desc':'CPI-U: Shelter','role':'Optional shelter cost pressure context','priority':'optional'},
 {'id':'CUSR0000SAS4','desc':'CPI-U: Transportation services','role':'Optional mobility cost context','priority':'optional'},
]
def fred_meta(series_id):
    if not FRED_API_KEY: return None, 'needs_review', 'Missing FRED_API_KEY; metadata not checked.'
    r = requests.get(f'{FRED_BASE}/series', params={'series_id':series_id,'api_key':FRED_API_KEY,'file_type':'json'}, timeout=30)
    if r.status_code != 200: return None, 'not_available', f'HTTP {r.status_code}: {r.text[:120]}'
    items = r.json().get('seriess', [])
    if not items: return None, 'not_available', 'Series not returned by FRED.'
    status = 'required_available' if series_id in {'MSPUS','CPIAUCSL','MEHOINUSA672N','MORTGAGE30US','CSUSHPINSA'} else 'optional_available'
    return items[0], status, items[0].get('notes','')
rows=[]
for c in fred_candidates:
    meta,status,notes=fred_meta(c['id'])
    rows.append({'source':'FRED','series_id_or_dataset':c['id'],'description':meta.get('title',c['desc']) if meta else c['desc'],'frequency':meta.get('frequency','') if meta else '', 'start_date':meta.get('observation_start','') if meta else '', 'end_date':meta.get('observation_end','') if meta else '', 'geography':'United States','role_in_story':c['role'],'ingestion_method':'FRED API observations endpoint','status':status,'notes':notes})
commute=[]
if FRED_API_KEY:
    r=requests.get(f'{FRED_BASE}/series/search', params={'search_text':'Mean Commute Time','api_key':FRED_API_KEY,'file_type':'json','limit':1000,'order_by':'search_rank','sort_order':'asc'}, timeout=30)
    if r.status_code==200:
        for item in r.json().get('seriess',[]):
            title=item.get('title','')
            if 'commute' in title.lower() or 'travel time' in title.lower(): commute.append(item)
    else: print('FRED commute search failed', r.status_code, r.text[:160])
commute_rows=[]
for item in commute[:200]:
    title=item.get('title','')
    commute_rows.append({'source':'FRED','series_id_or_dataset':item.get('id',''),'description':title,'frequency':item.get('frequency',''),'start_date':item.get('observation_start',''),'end_date':item.get('observation_end',''),'geography':'United States' if 'United States' in title else 'Subnational / needs parsing','role_in_story':'Commute/access burden candidate','ingestion_method':'FRED API search plus observations endpoint if selected','status':'needs_review','notes':'Candidate from controlled FRED series search for Mean Commute Time.'})
if commute_rows:
    write_csv(pd.DataFrame(commute_rows), INTERIM_DIR/'fred_commute_candidate_series.csv'); rows.extend(commute_rows[:25])
else:
    rows.append({'source':'FRED','series_id_or_dataset':'FRED release 415 / Mean Commute Time search','description':'FRED-hosted ACS mean commute-time series search','frequency':'','start_date':'','end_date':'','geography':'United States and subnational candidates','role_in_story':'Commute/access burden','ingestion_method':'FRED API series search; fallback to Census ACS API','status':'needs_review','notes':'No FRED commute candidates returned by controlled search, or key missing.'})
rows += [
 {'source':'Census ACS API','series_id_or_dataset':'acs/acs1 B08013_001E + B08012_001E','description':'Aggregate travel time to work divided by workers denominator for mean commute minutes','frequency':'Annual ACS 1-year','start_date':'2005 candidate','end_date':'latest available candidate','geography':'United States and large geographies','role_in_story':'National commute/access trend fallback','ingestion_method':'Controlled Census API requests with explicit variables and geographies','status':'needs_review','notes':'Confirm labels, universe, available years, and 2020 interruption before final use.'},
 {'source':'Census ACS API','series_id_or_dataset':'acs/acs5 B08013_001E + B08012_001E + B25077_001E + B19013_001E','description':'Selected county snapshot for commute, median home value, and median household income','frequency':'ACS 5-year release','start_date':'latest available','end_date':'latest available','geography':'Selected counties approximating case geographies','role_in_story':'Housing + access geographic lens','ingestion_method':'Controlled Census API requests for selected counties only','status':'needs_review','notes':'County proxies are not full metro areas; use only if clearly labeled.'},
]
cols=['source','series_id_or_dataset','description','frequency','start_date','end_date','geography','role_in_story','ingestion_method','status','notes']
inventory=pd.DataFrame(rows)[cols]
write_csv(inventory, INTERIM_DIR/'data_inventory.csv')
inventory.head(20)

wrote data/interim/data_inventory.csv (10 rows)


,source,series_id_or_dataset,description,frequency,start_date,end_date,geography,role_in_story,ingestion_method,status,notes
0,FRED,MSPUS,Median Sales Price of Houses Sold for the Unit...,,,,United States,Long-run housing sticker-price anchor,FRED API observations endpoint,needs_review,Missing FRED_API_KEY; metadata not checked.
1,FRED,CPIAUCSL,Consumer Price Index for All Urban Consumers: ...,,,,United States,Inflation adjustment baseline,FRED API observations endpoint,needs_review,Missing FRED_API_KEY; metadata not checked.
2,FRED,MEHOINUSA672N,Real Median Household Income in the United States,,,,United States,Income / ability-to-pay context,FRED API observations endpoint,needs_review,Missing FRED_API_KEY; metadata not checked.
3,FRED,MORTGAGE30US,30-Year Fixed Rate Mortgage Average in the Uni...,,,,United States,Financing context,FRED API observations endpoint,needs_review,Missing FRED_API_KEY; metadata not checked.
4,FRED,CSUSHPINSA,S&P CoreLogic Case-Shiller U.S. National Home ...,,,,United States,Repeat-sales home price index context,FRED API observations endpoint,needs_review,Missing FRED_API_KEY; metadata not checked.
5,FRED,CUSR0000SAH1,CPI-U: Shelter,,,,United States,Optional shelter cost pressure context,FRED API observations endpoint,needs_review,Missing FRED_API_KEY; metadata not checked.
6,FRED,CUSR0000SAS4,CPI-U: Transportation services,,,,United States,Optional mobility cost context,FRED API observations endpoint,needs_review,Missing FRED_API_KEY; metadata not checked.
7,FRED,FRED release 415 / Mean Commute Time search,FRED-hosted ACS mean commute-time series search,,,,United States and subnational candidates,Commute/access burden,FRED API series search; fallback to Census ACS...,needs_review,No FRED commute candidates returned by control...
8,Census ACS API,acs/acs1 B08013_001E + B08012_001E,Aggregate travel time to work divided by worke...,Annual ACS 1-year,2005 candidate,latest available candidate,United States and large geographies,National commute/access trend fallback,Controlled Census API requests with explicit v...,needs_review,"Confirm labels, universe, available years, and..."
9,Census ACS API,acs/acs5 B08013_001E + B08012_001E + B25077_00...,"Selected county snapshot for commute, median h...",ACS 5-year release,latest available,latest available,Selected counties approximating case geographies,Housing + access geographic lens,Controlled Census API requests for selected co...,needs_review,County proxies are not full metro areas; use o...
